# Request lifecycle (DNS→TLS→LB) & API versioning

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 24 §24.2–§24.3 · type: concept-lab*

**One-line promise:** trace one HTTPS request through DNS → TLS → load balancer → app server,
name the cost at each hop, *see* why in-memory state breaks horizontal scaling, and evolve an API
through a non-breaking change and a breaking one.

## 🧠 Why this matters

Type a URL and a surprising amount happens before any code of yours runs: a **DNS** lookup turns
the name into an IP, a **TLS** handshake secures the connection, a **load balancer** picks one of
several identical app servers, and only then does your handler execute. Knowing the path
demystifies most latency and "it's down for *some* users" incidents you'll ever debug.

That same load balancer is *why statelessness matters*: if any server can handle any request, you
scale by adding servers and survive one dying. Keep important state in a server's memory and you
lose that — requests must return to that exact box. Finally, your API is a **contract**: you'll
need to change it without breaking clients, which is what versioning is for. We simulate all of it
offline — no real sockets, resolvers, or certs.

## Objectives & prereqs

**By the end you can:**
- Walk an HTTPS request hop by hop and say where the **latency** goes (DNS, TLS, LB, app, data).
- Explain a stale-DNS incident and why caching makes it "down for some users."
- Show, with a toy round-robin load balancer, why **in-memory session state** breaks horizontal
  scaling — and why stateless handlers don't.
- Evolve an API safely: add an **optional** field (non-breaking) vs **rename/remove** one
  (breaking), and watch a pinned client fail on the breaking change.

**Prereqs:** notebook **24-01** (HTTP semantics & idempotency). No API key, no network — DNS/TLS/LB
are *simulated*. (Serialization here foreshadows gRPC/protobuf in Ch 26.)

In [ ]:
# --- Setup: imports, env, and the MOCK switch ---------------------------------
# stdlib only (+ python-dotenv from requirements.txt). Nothing leaves this process:
# every hop is a synthetic latency number; there are no real sockets or certs.
import os
import json
import random
from dataclasses import dataclass

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# No live path exists in this concept-lab, so MOCK stays 1 and the simulation runs
# free and deterministically (seeded) for every reader.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"
random.seed(2402)  # makes the synthetic latencies and LB routing reproducible

print(f"MOCK mode: {MOCK}  | DNS/TLS/LB are SIMULATED — no network egress")

## 1 · 🧠 The request lifecycle, hop by hop (§24.2)

Mirror the book's figure: **DNS → TLS → Load Balancer → App server → Data layer → back**. Each
stage costs time. The numbers below are an illustrative *latency budget* (typical-order
milliseconds), printed as a labeled trace so the path is concrete. They're representative, not a
benchmark.

In [ ]:
# A synthetic latency budget (ms). Real numbers vary wildly; the SHAPE is the lesson:
# the network hops (DNS, TLS) and the data layer usually dominate, not your handler.
LIFECYCLE = [
    ("DNS resolve",   "name -> IP; cached at many layers (often ~0ms on a hit)", 18),
    ("TCP + TLS",     "handshake sets up encryption (the 's' in https)",         35),
    ("Load balancer", "picks a healthy app server, forwards the request",         2),
    ("App server",    "YOUR handler runs: validate, route, orchestrate",          8),
    ("Data layer",    "DB / cache / vector store round-trip",                    25),
    ("Response back",  "serialize + return over the wire",                        12),
]
total = 0
print(f"{'stage':<15}{'ms':>5}   what happens")
print("-" * 64)
for stage, what, ms in LIFECYCLE:
    total += ms
    print(f"{stage:<15}{ms:>5}   {what}")
print("-" * 64)
print(f"{'TOTAL':<15}{total:>5} ms (illustrative)")
print("\nNote: a first request pays DNS+TLS; warm connections reuse them and get much faster.")

**What you just saw.** Most of the wall-clock time lives *outside* your handler — in the network
setup (DNS+TLS) and the data round-trip. That's why "make the Python faster" is often the wrong
first move: connection reuse (keep-alive), caching, and cutting data round-trips usually win more.

## 2 · A stale DNS record → "down for some users" (§24.2)

DNS answers are **cached** with a TTL at many layers (OS, resolver, app). When you move a service
to a new IP, caches that haven't expired keep sending users to the **old** address — so it's broken
for *some* people and fine for others, the classic confusing incident. We simulate a cache with a
TTL; no real resolver is touched.

In [ ]:
class DnsCache:
    """A toy resolver cache. Entries are (ip, ttl_left). On expiry it re-resolves
    from the authoritative record; until then it serves the stale answer."""
    def __init__(self, authoritative_ip):
        self.authoritative = authoritative_ip   # the CURRENT true record
        self.cached_ip = authoritative_ip
        self.ttl_left = 3

    def set_authoritative(self, new_ip):
        self.authoritative = new_ip              # ops moves the service...

    def resolve(self):
        if self.ttl_left <= 0:                   # cache expired -> pick up the new record
            self.cached_ip = self.authoritative
            self.ttl_left = 3
        self.ttl_left -= 1
        return self.cached_ip

cache = DnsCache("10.0.0.1")
cache.set_authoritative("10.0.0.9")             # service moved to a new IP

print("Resolutions after the IP change (TTL must expire before clients see it):")
for i in range(5):
    ip = cache.resolve()
    status = "STALE -> hits dead box" if ip != cache.authoritative else "fresh -> works"
    print(f"  lookup {i}: {ip}   ({status})")

**What you just saw.** Until the TTL expires, cached clients keep resolving to the **old** IP and
fail, while clients with a cold cache get the new one and succeed — "down for some users." The fix
in practice: lower TTLs *before* a planned move, and don't treat DNS as instant.

## 3 · 🔧 A toy round-robin load balancer (§24.2)

A load balancer sits in front of N identical servers and spreads requests across them — which is
what lets you scale **horizontally** (add more identical boxes) and survive one dying. Here's the
simplest policy, round-robin.

In [ ]:
@dataclass
class Server:
    name: str
    handled: int = 0

class LoadBalancer:
    def __init__(self, servers):
        self.servers = servers
        self._i = 0
    def route(self):
        s = self.servers[self._i % len(self.servers)]
        self._i += 1
        s.handled += 1
        return s

lb = LoadBalancer([Server("app-a"), Server("app-b"), Server("app-c")])
print("Routing 9 requests round-robin:")
for n in range(9):
    print(f"  request {n} -> {lb.route().name}")
print("\nspread:", {s.name: s.handled for s in lb.servers})

## 4 · 🔮 Predict: in-memory session state behind a load balancer

The handler below stores the user's session **in the server's memory**. The load balancer will
route the *same* client to a *different* server on the next request.

Before running the next cell, predict: when the client logs in on `app-a` and then their next
request lands on `app-b`, **is the session found?** What does that imply for adding/removing
servers?

In [ ]:
# Each server keeps its OWN in-memory session table — the anti-pattern.
class StatefulServer(Server):
    def __init__(self, name):
        super().__init__(name)
        self.sessions = {}                     # state trapped in THIS process's RAM
    def login(self, user):
        self.sessions[user] = {"user": user, "cart": ["item-1"]}
    def get_session(self, user):
        return self.sessions.get(user)         # None if this box never saw the login

servers = [StatefulServer("app-a"), StatefulServer("app-b"), StatefulServer("app-c")]
lb2 = LoadBalancer(servers)

s_login = lb2.route(); s_login.login("alice")            # login lands on one server
print(f"alice logged in on {s_login.name}")
s_next = lb2.route()                                     # next request -> different server
print(f"next request routed to {s_next.name}")
print("session found there? ->", s_next.get_session("alice"))
assert s_next.get_session("alice") is None
print("\nState was trapped on", s_login.name, "-> the other servers can't see it.")

**What you just saw.** The login lived only in `app-a`'s memory, so when the LB routed the next
request to `app-b`, the session was **gone**. Now you can't freely add/remove servers (requests
must return to the exact box that holds the state), and one server dying drops every session it
held — you've lost the easy scaling.

The stateless fix: keep per-request state **in the request** (e.g. a signed token) and shared state
**in the data layer** (Ch 30), not in process memory. Then *any* server can serve *any* request.

In [ ]:
# Stateless: session lives in a shared store (stand-in for Redis/DB, Ch 30).
shared_sessions = {}                            # the data layer, reachable by ALL servers

class StatelessServer(Server):
    def login(self, user):
        shared_sessions[user] = {"user": user, "cart": ["item-1"]}
    def get_session(self, user):
        return shared_sessions.get(user)        # same answer on every server

servers = [StatelessServer("app-a"), StatelessServer("app-b"), StatelessServer("app-c")]
lb3 = LoadBalancer(servers)

lb3.route().login("alice")                      # login on whichever server
found = lb3.route().get_session("alice")        # next request, different server
print("session found on a different server now? ->", found)
assert found is not None
print("Any server can serve any request -> horizontal scaling restored.")

## 5 · Serialization & content types (§24.3)

Data crossing the network is **serialized** to bytes and deserialized on the other side. **JSON** is
the web's lingua franca — human-readable, universal, the right default. Compact binary formats
(Protocol Buffers with gRPC, Ch 26) trade legibility for size/speed. The `Content-Type` header
announces which format is on the wire so the other side knows how to parse it.

In [ ]:
obj = {"id": 42, "role": "assistant", "tokens": 137, "ok": True}

# JSON: text, readable, self-describing field names on every message.
as_json = json.dumps(obj).encode("utf-8")

# A compact binary-ish encoding (stand-in for protobuf): drop field names, pack values
# positionally. Real protobuf is more sophisticated; the trade-off is the point.
def pack_compact(o):
    parts = [str(o["id"]), o["role"][:1], str(o["tokens"]), "1" if o["ok"] else "0"]
    return "\x1f".join(parts).encode("utf-8")     # 0x1f = unit separator

as_binary = pack_compact(obj)

print(f"JSON   ({len(as_json):>3} bytes, Content-Type: application/json):")
print("   ", as_json)
print(f"binary ({len(as_binary):>3} bytes, Content-Type: application/x-msg):")
print("   ", as_binary)
print(f"\nbinary is {100 * (1 - len(as_binary) / len(as_json)):.0f}% smaller — but you "
      "can't read it, and both sides must agree on the schema (foreshadows gRPC, Ch 26).")

**What you just saw.** The binary form is smaller but opaque and schema-coupled; JSON is bigger but
self-describing and debuggable. Default to JSON; reach for binary only when size/latency at scale
justifies the cost — and let `Content-Type` keep the two sides honest about which is in use.

## 6 · 🔧 Versioning: evolve the contract without breaking clients (§24.3)

Your API is a contract. The safe rule: **add, never mutate or remove**. Adding an *optional* field
or a new endpoint is non-breaking — old clients ignore what they don't know. *Renaming* or
*removing* a field is breaking — a pinned client that reads the old name fails. When you must
break, do it behind a **new version** (`/v1` → `/v2`).

In [ ]:
# A client pinned to v1 reads exactly these fields and nothing else.
def pinned_v1_client(payload):
    """Reads 'id' and 'text'. Breaks if 'text' disappears."""
    return f"[client] message {payload['id']}: {payload['text']!r}"

# v1 of the endpoint.
def endpoint_v1(text):
    return {"id": 1, "text": text}

# v1.1 — NON-breaking: ADD an optional field. The pinned client is unaffected.
def endpoint_v1_1(text):
    return {"id": 1, "text": text, "lang": "en"}   # new optional field

print("v1     :", pinned_v1_client(endpoint_v1("hello")))
print("v1.1   :", pinned_v1_client(endpoint_v1_1("hello")), " <- still works: added field ignored")

> ⚠️ **Pitfall — a breaking change on an *unversioned* API.** Rename `text` → `content` in place
> and *every* integration that read `text` breaks silently, all at once. The cell below 🔮 *predict*
> first: what happens when the pinned v1 client meets the renamed field? Then see the safe path:
> ship the rename as **`/v2`** and leave `/v1` intact.

In [ ]:
# v2 — BREAKING: rename 'text' -> 'content'. Correct ONLY because it's a new version.
def endpoint_v2(text):
    return {"id": 1, "content": text}              # 'text' is gone

# A v1 client hitting v2's shape blows up (it looks for 'text'):
try:
    pinned_v1_client(endpoint_v2("hello"))
except KeyError as e:
    print("pinned v1 client against v2 shape -> KeyError:", e, " <- THIS is the breakage")

# The safe deployment: keep BOTH surfaces. Old clients stay on /v1; new clients opt into /v2.
ROUTES = {
    "/v1/messages": endpoint_v1_1,   # unchanged contract, still served
    "/v2/messages": endpoint_v2,     # new shape, new path
}
print("\n/v1 still serves old clients:", pinned_v1_client(ROUTES["/v1/messages"]("hi")))
print("/v2 serves new clients     :", ROUTES["/v2/messages"]("hi"))
print("\nAdd, never mutate: version the breaking change, keep /v1 alive through a deprecation window.")

**What you just saw.** The in-place rename **broke** the pinned client (`KeyError: 'text'`). Moving
the same change to `/v2` while keeping `/v1` intact let old and new clients coexist — the only safe
way to make a breaking change to a contract real clients depend on.

## 🎯 Senior lens

Two judgment calls hide in this chapter. First, **latency**: when something is slow, a senior looks
at the *whole* lifecycle before touching code — a 200ms p99 is far more often a cold TLS handshake,
a chatty data layer, or a missing cache than slow Python; optimize the dominant hop, not the
convenient one. Second, **statelessness is a scaling decision, not a code-style preference**: the
moment you stash session or run-state in process memory you've quietly coupled a user to a machine,
forfeited free horizontal scaling, and signed up for sticky sessions and lost data on restart. Keep
handlers stateless and checkpoint agent run-state to the data layer (Ch 14, Ch 30). And treat the
**published API shape as immutable** — adding is free, mutating is a breaking change wearing a
disguise; version it.

## Recap

- An HTTPS request flows **DNS → TLS → load balancer → app → data → back**; the network setup and
  data round-trip usually dominate latency, not your handler.
- DNS answers are **cached with a TTL**; stale caches cause "down for *some* users" after an IP
  move — lower TTLs before planned changes.
- A load balancer spreads requests for **horizontal scaling**; that only works if handlers are
  **stateless** — keep per-request state in the request, shared state in the data layer.
- **JSON** is the default serialization; binary (protobuf/gRPC, Ch 26) trades legibility for size.
  `Content-Type` announces the format.
- Evolve APIs by **adding** (optional fields, new endpoints = non-breaking); **rename/remove only
  behind a new version** (`/v1` → `/v2`). A breaking change to an unversioned API breaks everyone.

## Exercises

Predict the result before running each.

1. **Version a field rename safely.** Starting from `endpoint_v1`, you must rename `text` →
   `content`. Implement it so a v1 client *and* a v2 client both keep working. Then add a `/v2`
   client and confirm both pass.
2. **Least-connections LB.** Replace round-robin with a policy that routes to the server with the
   fewest in-flight requests. Give two servers a slow handler and one a fast one; predict the spread
   versus round-robin.
3. **Sticky sessions, honestly.** Make the load balancer route a given user to the *same* stateful
   server every time (hash the user id). It "works" — now list two reasons a senior still prefers
   the stateless design from §4.
4. **DNS TTL trade-off.** Change `DnsCache`'s TTL to 1 and re-run §2. How does a *shorter* TTL change
   the incident window? What does it cost (hint: more lookups)? When would you deliberately raise it?

In [ ]:
# Exercise 1 — your code here

In [ ]:
# Exercise 2 — your code here

In [ ]:
# Exercise 3 — your code here

In [ ]:
# Exercise 4 — your code here

## Next

- ⬅️ **Previous:** [`24-01-http-and-idempotency.ipynb`](./24-01-http-and-idempotency.ipynb) — HTTP
  semantics, status codes, and the idempotency key.
- ➡️ **Next chapter (Ch 25):** you build a real **FastAPI** service that honors these contracts —
  correct status codes, a `/v1` surface, and an idempotent `POST /runs`. See
  [`templates/fastapi-agent-service/`](../../../templates/fastapi-agent-service/).
- 🔭 **Looking ahead:** the JSON-vs-binary trade-off returns as **gRPC/protobuf** in **Ch 26**;
  statelessness and checkpointing recur in **Ch 14** and **Ch 29–30**.
- 🏗️ **Capstone:** the lifecycle, statelessness, and `/v1` versioning here are the operating
  assumptions the capstone service is built on.
- See the book **§24.2–§24.3** for the request-lifecycle figure, statelessness, and the versioning
  pitfall.